[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/00_p03_reductions.ipynb)

# 🔵 Prerequisite 3: Reductions Along Dims

Softmax, LayerNorm, BatchNorm, RMSNorm, gradient clipping, quantization scales, and GRPO advantages all **reduce** a tensor along some dimension (sum, mean, max, variance, norm), then usually **broadcast the result back** against the original.

Two choices decide whether that works: **which `dim`** you reduce over, and whether you keep it with **`keepdim=True`**. A third choice, **biased vs unbiased variance**, silently changes your numbers.

### What you'll learn
1. What `dim` means: the dimension that **disappears**
2. `keepdim=True` so the result broadcasts back
3. `max` / `min` return `(values, indices)`; `amax` / `argmax`
4. `var` / `std`: the unbiased default and when you need `unbiased=False`
5. Reducing over several dims at once
6. Norms, and reducing across a *list* of tensors
7. Cumulative sums, inclusive and exclusive

### Exercises
| # | Function | Shows up in |
|---|----------|-------------|
| 1 | `stable_shift(x)` | #02 Softmax, #25 Flash Attention |
| 2 | `normalize_rows(x)` | #02 Softmax, #12 Linear Attention |
| 3 | `channel_stats(x)` | #07 BatchNorm |
| 4 | `masked_mean(x, mask)` | #41 OPD Loss, sequence pooling |
| 5 | `global_l2_norm(tensors)` | #21 Gradient Clipping |
| 6 | `exclusive_cumsum(x, dim)` | #32 Top-p Sampling |

### Rules
- **No Python loops** over tensor elements or channels. (Exercise 5 may loop over the *list* of tensors.)
- Do **not** use `torch.softmax`, `F.normalize`, `torch.nn.utils.clip_grad_norm_`, or `torch.nn.functional` normalization layers.

### How to use this notebook
1. **Run the lesson cells** (sections 1, 2, …) from top to bottom. They are demos: change them and re-run.
2. **Do the exercises in order.** Each one has its own `✏️ YOUR IMPLEMENTATION HERE` cell, followed by a debug cell for checking shapes before you submit.
3. **Submit at any time** with the `check("p03_reductions")` cell at the bottom. It grades every exercise at once, and each test is named after its exercise, so a partial score shows which ones are left.
4. **Stuck?** `hint("p03_reductions")` gives one hint per exercise; `00_p03_reductions_solution.ipynb` has the reference code.
5. **Already comfortable with this skill?** Prerequisites are optional. Skip straight to the problems listed in **🔓 What this unlocks** at the bottom.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

## 1. `dim` is the dimension that disappears

For a `(B, N)` tensor, `x.sum(dim=-1)` adds **along** the last dim, giving one number per row: shape `(B,)`.
`x.sum(dim=0)` adds down the columns, giving one number per column: shape `(N,)`.

In [ ]:
x = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
print(x.sum(dim=-1))   # tensor([ 6., 15.])     one per row
print(x.sum(dim=0))    # tensor([5., 7., 9.])   one per column
print(x.sum())         # tensor(21.)            everything -> 0-d tensor

## 2. `keepdim=True` to broadcast back

Without `keepdim`, a `(B, N)` → `(B,)` result can't be subtracted from `x` row-wise. Broadcasting aligns from the **right**, so `(B,)` lines up with `N`.
With `keepdim=True` you get `(B, 1)`, which broadcasts correctly. *(See P2 Broadcasting.)*

In [ ]:
x = torch.randn(3, 3)
wrong = x - x.mean(dim=-1)                 # (3,3) - (3,) -> subtracts per COLUMN, silently!
right = x - x.mean(dim=-1, keepdim=True)   # (3,3) - (3,1) -> per row
print(right.mean(dim=-1))                  # ~0 for every row
print(wrong.mean(dim=-1))                  # not 0

## 3. `max` returns values *and* indices

`x.max(dim=...)` returns a named tuple `(values, indices)`. Use `.values`, or unpack it. `x.amax(dim=...)` returns only the values (and accepts several dims); `x.argmax(dim=...)` returns only the indices.

In [ ]:
x = torch.tensor([[3., 9., 1.],
                  [7., 2., 8.]])
vals, idx = x.max(dim=-1)
print(vals, idx)                           # tensor([9., 8.]) tensor([1, 2])
print(x.max(dim=-1, keepdim=True).values)  # (2, 1)
print(x.amax(dim=-1), x.argmax(dim=-1))

## 4. ⚠️ Variance: unbiased by default

`x.var()` and `x.std()` divide by `N - 1` by default (the *unbiased* sample estimate).
**Normalization layers divide by `N`**: LayerNorm, BatchNorm's normalization, and RMSNorm. Pass `unbiased=False` (or `correction=0`).
With small `N` the difference is large; tests will catch it.

In [ ]:
x = torch.tensor([1., 2., 3., 4.])
print(x.var())                    # 1.6667  divides by N-1 = 3
print(x.var(unbiased=False))      # 1.25    divides by N   = 4
print(((x - x.mean()) ** 2).mean())   # 1.25, the definition

## 5. Several dims at once

Pass a tuple: `x.mean(dim=(0, 2, 3))` reduces a `(B, C, H, W)` tensor to one value per channel `(C,)`. That is how BatchNorm2d computes its statistics.

In [ ]:
x = torch.randn(8, 3, 4, 4)
print(x.mean(dim=(0, 2, 3)).shape)                  # torch.Size([3])
print(x.amax(dim=(-2, -1), keepdim=True).shape)     # torch.Size([8, 3, 1, 1])

## 6. Norms and lists of tensors

`x.norm()` is the L2 norm of **all** elements: `sqrt(sum(x**2))`. `x.norm(dim=-1)` gives one norm per row.
To reduce across a **list** of differently-shaped tensors (like every parameter's gradient), reduce each one to a scalar, then combine the scalars. Never concatenate everything into one huge tensor.

In [ ]:
grads = [torch.randn(4, 4), torch.randn(10)]
per_tensor = torch.stack([g.pow(2).sum() for g in grads])  # one scalar each
print(per_tensor.sum().sqrt())
print(torch.cat([g.flatten() for g in grads]).norm())      # same value, but copies everything

## 7. Cumulative sums

`x.cumsum(dim)` keeps the shape and replaces each element with the running total up to and **including** it.
The **exclusive** version, the total of everything strictly **before** each position, is `x.cumsum(dim) - x`.

Top-p ("nucleus") sampling (#32) sorts the probabilities in descending order and keeps a token while the probability mass **before** it is still below `p`. The exclusive cumsum makes sure the token that crosses `p` is kept.

In [ ]:
p = torch.tensor([0.5, 0.3, 0.15, 0.05])     # sorted, descending
print(p.cumsum(dim=0))                       # [0.50, 0.80, 0.95, 1.00]  inclusive
print(p.cumsum(dim=0) - p)                   # [0.00, 0.50, 0.80, 0.95]  exclusive
print((p.cumsum(dim=0) - p) < 0.7)           # keep [True, True, False, False]: 0.5 + 0.3 >= 0.7

---
# ✏️ Exercises

Implement the six functions below. Each one has a debug cell you can use before you submit.

### Exercise 1: `stable_shift(x)`

Subtract each row's maximum from that row, so the largest value in every row becomes `0`.

- `x`: any shape; "row" means the **last** dim
- returns: same shape as `x`

*Softmax is invariant to this shift, and it prevents `exp` from overflowing (#02). Flash Attention does the same with a running max (#25).*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def stable_shift(x: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.tensor([[1., 1000., 3.],
                  [5., 2., 4.]])
print(stable_shift(x))
# expect [[-999.,    0., -997.],
#         [   0.,   -3.,   -1.]]

### Exercise 2: `normalize_rows(x)`

Scale each row so it sums to `1`.

- `x`: positive values, any shape; "row" means the **last** dim
- returns: same shape, with `out.sum(dim=-1) == 1` everywhere

*Softmax is `normalize_rows(exp(stable_shift(x)))`. Linear attention divides by a summed normalizer the same way (#12).*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def normalize_rows(x: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.tensor([[1., 1., 2.],
                  [3., 1., 0.]])
p = normalize_rows(x)
print(p)               # expect [[0.25, 0.25, 0.50], [0.75, 0.25, 0.00]]
print(p.sum(dim=-1))   # expect tensor([1., 1.])

### Exercise 3: `channel_stats(x)`

Compute per-channel mean and **biased** variance of a batch of images, as BatchNorm2d does in training mode.

- `x`: `(B, C, H, W)`
- returns: a tuple `(mean, var)`, each of shape `(C,)`, reducing over batch **and** spatial dims
- use the **biased** variance (divide by the number of elements, not by that number minus 1)

*#07 BatchNorm does the same for `(N, D)` inputs, reducing over dim 0 only.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def channel_stats(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    pass  # Replace this

In [ ]:
x = torch.zeros(2, 3, 2, 2)
x[:, 1] = 5.0                      # channel 1 is constant 5
x[0, 2], x[1, 2] = 1.0, 3.0        # channel 2 is half 1s, half 3s
mean, var = channel_stats(x)
print(mean)   # expect tensor([0., 5., 2.])
print(var)    # expect tensor([0., 0., 1.])   (unbiased would give 1.1429)

### Exercise 4: `masked_mean(x, mask)`

Average token vectors over the sequence, counting only the **valid** positions.

- `x`: `(B, S, D)`
- `mask`: `(B, S)`, either `bool` or `0/1` floats; `True` / `1` = valid
- returns: `(B, D)`, where `out[b] = mean of x[b, s] over valid s`
- if a sequence has **no** valid positions, return zeros for it (not `NaN`)

*#41 OPD averages a per-token loss over unmasked tokens with exactly this pattern.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def masked_mean(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.tensor([[[1., 10.], [3., 30.], [100., 100.]],
                  [[5., 50.], [7., 70.], [9., 90.]]])
mask = torch.tensor([[True, True, False],
                     [False, False, False]])
print(masked_mean(x, mask))
# expect [[2., 20.],    mean of the first two tokens
#         [0.,  0.]]    no valid tokens -> zeros

### Exercise 5: `global_l2_norm(tensors)`

Compute the L2 norm of **all** elements of **all** tensors in a list, as if they were concatenated into one long vector.

- `tensors`: a list of tensors with different shapes
- returns: a 0-d tensor, `sqrt(sum over every tensor of sum(t ** 2))`

*This is the total gradient norm in #21 Gradient Clipping.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def global_l2_norm(tensors: list[torch.Tensor]) -> torch.Tensor:
    pass  # Replace this

In [ ]:
ts = [torch.tensor([3., 0.]), torch.tensor([[0., 4.]])]
print(global_l2_norm(ts))   # expect tensor(5.)

### Exercise 6: `exclusive_cumsum(x, dim=-1)`

For each position, the sum of all elements **strictly before** it along `dim`.

- `x`: any shape
- returns: same shape; `out[..., 0] == 0` and `out[..., i] = x[..., 0] + … + x[..., i-1]`

*#32 Top-p keeps sorted token `i` while `exclusive_cumsum(sorted_probs)[i] < p`.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def exclusive_cumsum(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    pass  # Replace this

In [ ]:
print(exclusive_cumsum(torch.tensor([[1., 2., 3., 4.],
                                     [0.5, 0.25, 0.25, 0.]])))
# expect [[0., 1., 3., 6.],
#         [0., 0.5, 0.75, 1.]]

<!-- Generated from scripts/prereqs/content/p03_reductions.py by `make prereqs`. Edit that file instead of this notebook; manual edits are overwritten. -->
## ✅ Submit

Run the judge on **all** exercises in this notebook. Each test is labelled with the exercise it checks.

In [ ]:
from torch_judge import check, hint
check("p03_reductions")

In [ ]:
# 💡 Stuck? Uncomment for a nudge
# hint("p03_reductions")

---
## 🔓 What this unlocks

With **Reductions Along Dims** in your toolkit, you're ready for these problems:

| # | Problem | Difficulty | What you'll reuse | Links |
|:---:|---|:---:|---|---|
| #2 | Softmax | 🟢 Easy | `stable_shift` then `normalize_rows`: softmax is exactly these two steps around `exp` | [Open](02_softmax.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/02_softmax.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/02_softmax.ipynb" target="_blank">Colab</a> |
| #4 | LayerNorm | 🟡 Medium | Mean and **biased** variance over the last dim, with `keepdim=True` | [Open](04_layernorm.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/04_layernorm.ipynb" target="_blank">Colab</a> |
| #7 | BatchNorm | 🟡 Medium | `channel_stats`: batch statistics with `unbiased=False` | [Open](07_batchnorm.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb" target="_blank">Colab</a> |
| #42 | BatchNorm2d | 🟡 Medium | `channel_stats` is exactly its statistics step: reduce over `(0, 2, 3)` | [Open](42_batchnorm2d.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/42_batchnorm2d.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/42_batchnorm2d.ipynb" target="_blank">Colab</a> |
| #8 | RMSNorm | 🟡 Medium | `x.pow(2).mean(dim=-1, keepdim=True)`: the mean square inside RMSNorm | [Open](08_rmsnorm.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/08_rmsnorm.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/08_rmsnorm.ipynb" target="_blank">Colab</a> |
| #12 | Linear Attention | 🔴 Hard | Dividing by a summed normalizer, like `normalize_rows` | [Open](12_linear_attention.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb" target="_blank">Colab</a> |
| #21 | Gradient Clipping | 🟢 Easy | `global_l2_norm`: the total norm across all parameter gradients | [Open](21_gradient_clipping.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb" target="_blank">Colab</a> |
| #25 | Flash Attention | 🔴 Hard | `stable_shift` with a running max: the online-softmax trick | [Open](25_flash_attention.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb" target="_blank">Colab</a> |
| #32 | Top-k / Top-p Sampling | 🟡 Medium | `exclusive_cumsum` of the sorted probabilities decides which tokens top-p keeps | [Open](32_topk_sampling.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb" target="_blank">Colab</a> |
| #36 | INT8 Quantization | 🔴 Hard | Per-row `abs().amax(dim=1, keepdim=True)` for the quantization scale | [Open](36_int8_quantization.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb" target="_blank">Colab</a> |
| #38 | GRPO Loss | 🔴 Hard | Per-group mean and **biased** std of rewards for the advantages | [Open](38_grpo_loss.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb" target="_blank">Colab</a> |
| #41 | OPD Loss | 🔴 Hard | `masked_mean`: average a per-token loss over valid tokens only | [Open](41_opd_loss.ipynb) · <a href="https://github.com/duoan/TorchCode/blob/master/templates/41_opd_loss.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/41_opd_loss.ipynb" target="_blank">Colab</a> |

🗺️ See the full recommended order in <a href="https://github.com/duoan/TorchCode/blob/master/LEARNING_PATH.md" target="_blank">LEARNING_PATH.md</a>.